<a href="https://colab.research.google.com/github/bhagyoday-j/ML-Assignments/blob/main/TY-open-elective/Assingments/5_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Assignment : 5***
Web Page Phishig Detection

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("peyamowar/phishing-and-benign-websites")

print("Path to dataset files:", path)

100%|██████████| 858k/858k [00:00<00:00, 24.4MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/peyamowar/phishing-and-benign-websites/versions/1


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import os

# The path to the dataset was printed in the previous cell
dataset_path = '/root/.cache/kagglehub/datasets/peyamowar/phishing-and-benign-websites/versions/1'

print(f"Attempting to load data from: {dataset_path}")

try:
    file_name = 'phishing_and_benign_websites.csv'
    full_file_path = os.path.join(dataset_path, file_name)

    print(f"\nAttempting to load data from: {full_file_path}")
    df = pd.read_csv(full_file_path)
    print("Dataset loaded successfully!")
    print("Shape of the dataset:", df.shape)
    print("First 5 rows:\n", df.head())
    print("Column information:\n", df.info())

    # Correctly identify target variable and features
    # Based on df.head() and df.info(), 'Label' is the target. 'URLs' is the feature.
    y = df['Label']
    print("Target variable 'Label' identified.")

    # Encode target variable
    global label_encoder # Declare as global for later use in deployment
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    print(f"Target variable encoded. Original classes: {label_encoder.classes_}")

    # Feature Engineering: Extracting simple numerical features from URLs
    # For a basic Logistic Regression model, let's start with URL length as a feature.
    # More sophisticated features would involve parsing URLs (e.g., domain length, path length, number of dots, etc.)
    df['url_length'] = df['URLs'].apply(len)
    X = df[['url_length']] # X should be a DataFrame or 2D array for sklearn models
    print(f"Features created: {list(X.columns)}")

    global feature_names # Declare as global for later use in deployment
    feature_names = X.columns.tolist() # Store feature names for consistency during inference

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

    print(f"Data split: X_train shape {X_train.shape}, X_test shape {X_test.shape}")

    # Initialize and train Logistic Regression model
    global trained_model # Declare as global for later use in deployment
    trained_model = LogisticRegression(max_iter=1000, solver='liblinear')
    trained_model.fit(X_train, y_train)

    # Evaluate the model
    y_pred = trained_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=label_encoder.classes_)

    print(f"\nLogistic Regression Model Accuracy: {accuracy:.4f}")
    print("\nClassification Report:\n", report)

    print("\nModel training complete. The 'trained_model' object is now available for deployment.")

except FileNotFoundError:
    print(f"Error: The file {full_file_path} was not found. Please check the dataset path and file name.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Attempting to load data from: /root/.cache/kagglehub/datasets/peyamowar/phishing-and-benign-websites/versions/1

Attempting to load data from: /root/.cache/kagglehub/datasets/peyamowar/phishing-and-benign-websites/versions/1/phishing_and_benign_websites.csv
Dataset loaded successfully!
Shape of the dataset: (38800, 2)
First 5 rows:
                                           URLs   Label
0                     http://www.wmmayhem.com/  Benign
1  http://www.ballymenaunitedyouthacademy.com/  Benign
2              http://www.brusselsgaybars.com/  Benign
3          http://www.sportsbettingtennis.net/  Benign
4                         http://www.i29.mobi/  Benign
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38800 entries, 0 to 38799
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   URLs    38800 non-null  object
 1   Label   38800 non-null  object
dtypes: object(2)
memory usage: 606.4+ KB
Column information:
 None
Target variable

In [6]:
from flask import Flask, request, jsonify
from flask_ngrok import run_with_ngrok
import time
import sys
import os

# Initialize Flask app
app = Flask(__name__)
run_with_ngrok(app) # Start ngrok when app is run

# Store memory usage and inference times
# This can be made more robust for concurrent requests if needed, but for simple evaluation it's fine.
memory_usage_log = []
inference_time_log = []

# Helper function to get current memory usage (in MB)
def get_memory_usage():
    return sys.getsizeof(trained_model) / (1024 * 1024) # Placeholder, ideally measure process memory

@app.route('/')
def home():
    return "Model API is running! Send POST requests to /predict."

@app.route('/predict', methods=['POST'])
def predict():
    start_time = time.time()
    data = request.get_json(force=True)

    if 'url' not in data:
        return jsonify({'error': 'Missing "url" field in request.'}), 400

    url = data['url']

    try:
        # Feature extraction - must be consistent with training
        url_length = len(url)
        features = pd.DataFrame([[url_length]], columns=feature_names)

        # Make prediction
        prediction_encoded = trained_model.predict(features)[0]
        prediction_proba = trained_model.predict_proba(features)[0]

        # Decode prediction
        predicted_label = label_encoder.inverse_transform([prediction_encoded])[0]

        # Prepare probabilities for output
        probabilities = {label: prob for label, prob in zip(label_encoder.classes_, prediction_proba)}

        end_time = time.time()
        inference_time = (end_time - start_time) * 1000 # in milliseconds

        # Log metrics (simplistic for demonstration)
        memory_usage = get_memory_usage()
        inference_time_log.append(inference_time)
        memory_usage_log.append(memory_usage)

        return jsonify({
            'prediction': predicted_label,
            'probabilities': probabilities,
            'inference_time_ms': inference_time,
            'model_memory_usage_mb': memory_usage # This is just the model object size, not process memory
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500


# This part needs to be run in a separate cell or handled carefully in Colab if not using run_with_ngrok properly
# For simplicity, run_with_ngrok already starts the app in a new thread.
# app.run()

print("Flask app with prediction endpoint '/predict' is ready. Ngrok tunnel will start upon execution of this cell.")
print("You can send POST requests with JSON payload like: {'url': 'http://example.com'}")

ModuleNotFoundError: No module named 'flask_ngrok'

In [7]:
!pip install flask_ngrok